<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T12-L02-rainfall-runoff-honestly/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T12-L02-rainfall-runoff-honestly/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T12-L02-rainfall-runoff-honestly/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have. The cell below fetches the files it needs beside it, and does
nothing where they are already present. On Kaggle, switch Internet on in the notebook's
settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/camels_aus_v2_two_basins_1985_2014.npz"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T12-L02-rainfall-runoff-honestly/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T12-L02 · Rainfall to runoff, honestly

Forecast river flow from rainfall in two real Australian catchments, then find out how
easily a good-looking score can be a mistake: the wrong benchmark, the wrong split, the
wrong years, the wrong part of the hydrograph.

**You will build:** Nash-Sutcliffe and Kling-Gupta efficiency, a day-of-year climatology
benchmark, a small conceptual rainfall-runoff model (a four-parameter model in the GR4J family: a
soil-moisture store, a routing store and two unit hydrographs), a deterministic calibration
search, a lagged-rainfall regression baseline, a chronological hold-out, and flow-band
scores. **Then you will measure** what a random split, in-sample scoring and a single
headline number each hide.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no network on the required path ·
**Prerequisite:** T00-L01-the-8gb-track.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- slice a daily numpy series by year and compute means, including a mean for each day of the year
- explain train, validation and test sets, and why scoring a model on its own training data overstates its skill
- fit a linear regression with numpy, or solve a small linear system with np.linalg.solve
- say what a catchment, streamflow, rainfall and evapotranspiration are, and read a hydrograph (flow against time)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Technical note: Inherent benchmark or not? Comparing Nash-Sutcliffe and Kling-Gupta efficiency scores](https://hess.copernicus.org/articles/23/4323/2019/) · Knoben, Freer and Woods (Hydrology and Earth System Sciences) · *free* — what NSE and KGE reward, why a benchmark is needed to read either score, and why mean flow scores NSE 0 but KGE near -0.41 (exercises 1 and 2)
- [airGR: the INRAE GR hydrological models](https://hydrogr.github.io/airGR/) · INRAE Catchment Hydrology Research Group · *free* — the documentation of GR4J, the four-parameter daily lumped model family that this lesson's four-parameter model is a simplification of
- [Forecasting: Principles and Practice, 3rd edition: Evaluating point forecast accuracy](https://otexts.com/fpp3/toolbox.html#sec-accuracy) · Rob J. Hyndman and George Athanasopoulos · *free* — section 5.8: training versus test data for time series, and accuracy compared against simple benchmarks such as the mean and naive methods, the logic of exercises 3 and 7
- [An Introduction to Statistical Learning](https://www.statlearning.com/) · James, Witten, Hastie and Tibshirani (Springer; free PDF from the authors) · *free* — section 6.2.1 on ridge regression (exercise 8), and section 5.1.1 on the validation-set approach and what a single split can and cannot tell you
- [Rainfall-Runoff Modelling: The Primer, 2nd edition](https://www.wiley.com/en-us/rainfall-runoff-modelling-the-primer-2nd-edition-p-9780470714591) · Keith J. Beven (Wiley-Blackwell) · *paid* — chapter 2 on how rainfall-runoff models evolved (2.3, the unit hydrograph), chapter 4 on data-based transfer-function models, chapter 7 on parameter estimation and predictive uncertainty

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
from pathlib import Path
from typing import Callable

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("nse",),
    "exercise 2": ("kge",),
    "exercise 3": ("doy_climatology", "benchmark_skill"),
    "exercise 4": ("gr4j_unit_hydrographs",),
    "exercise 5": ("gr4j_simulate",),
    "exercise 6": ("pattern_search",),
    "exercise 7": ("chronological_split", "holdout_evaluation"),
    "exercise 8": ("lagged_matrix", "ridge_fit"),
    "exercise 9": ("flow_band_scores",),
}


def _named(labels: list[str]) -> str:
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


@contextlib.contextmanager
def _time_limit(seconds: float, message: str = ""):
    """Turn a run that never ends into a failed check instead of a frozen notebook.

    Uses the operating system's alarm, so it works where that exists (Linux, macOS, Colab) and is
    skipped, harmlessly, where it does not. A limit inside another (or inside the grader's own)
    never switches the outer one off: the outer keeps its deadline if it is due first and gets
    back what is left of its time, and its repeat interval, when the inner one ends.
    """
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    message = message or (f"still running after {seconds:g} s: look for a loop whose "
                          "condition never changes")
    started = time.monotonic()
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False,
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"] = owner                               # the frame running the `with`

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in this guard's own
        while frame is not None and frame is not state["owner"]:     # set-up, exit or clean-up (or
            if (frame is state["me"] or frame.f_code in (_fire.__code__, _arm.__code__)    # in code
                    or (frame.f_back is state["owner"] and frame.f_globals is vars(contextlib))):
                return True                              # run from contextlib's __exit__ under the `with`)?
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        if state["next"] is not None and now >= state["next"] - 0.005:
            state["fired"] = True
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            raise _OutOfTime(message, state)
        _arm(now)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        raise AssertionError(message) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        raise AssertionError(message)


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message -- which names the likely mistake -- and the notebook carries on. A demo names the
    exercises it `needs`: until each has passed its check, the demo says which one it is
    waiting for and skips.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(180):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The data: two real catchments, daily records

`assets/camels_aus_v2_two_basins_1985_2014.npz` holds daily catchment-average rainfall,
potential evapotranspiration (PET) and observed river flow for two Australian catchments,
taken from the CAMELS-AUS v2 dataset (licence, citation, exact files and how the two
catchments were chosen: `assets/SOURCE.md` and `claims.yaml`). It is committed with the lesson
and read with **no network access**. All three series are licensed CC BY 4.0; rain and PET come
from SILO (Queensland Government) and flow from the Bureau of Meteorology (`claims.yaml`:
`silo-licence`, `bom-hrs-licence`).

The two catchments come from a written selection rule that looks only at the dataset's own
catalogue, never at a model result: gauges flagged as Hydrologic Reference Stations, complete
records, a size band, and then the most winter-dominant and the most summer-dominant rainfall
regime that qualify (`assets/SOURCE.md`). The PET series is the one the dataset's authors
recommend as forcing; for rain they recommend a different gridded product, which this lesson
does not use for licensing reasons (`claims.yaml`: `camels-aus-forcing-recommendation`). Flow is in
**millimetres per day of depth over the catchment**, the same unit as rain
(`claims.yaml`: `camels-aus-streamflow-units`), so a runoff ratio is one mean divided by the
other. Nothing here is synthetic.

**Choices made after looking.** You should know four choices this lesson's authors made after
seeing some of this data. The selection rule's thresholds were set after looking at the
catalogue attributes of the eligible catchments (before any model was run on the chosen two),
and nearby thresholds pick other rivers. An early draft used a different PET series from the
same dataset and switched to the recommended one after some model runs. The rain series was
swapped for the dataset's SILO rainfall after the whole lesson had been run on the recommended
one (for the licence, not the scores; `assets/SOURCE.md` says what changed). The regression
baseline's 30 days of lagged rain were picked by comparing 7, 30 and 90 days on these records;
section 10 re-runs that comparison in front of you. `assets/SOURCE.md` has the details. And two
rivers are two examples, not a sample: nothing here says how large these gaps are elsewhere.

In [ ]:
ASSET_NAME = "camels_aus_v2_two_basins_1985_2014.npz"


def _asset_path() -> Path:
    """Locate the committed data next to this file or its parent lesson directory."""
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / ASSET_NAME, here.parent / "assets" / ASSET_NAME,
                      Path.cwd() / "assets" / ASSET_NAME,
                      Path.cwd().parent / "assets" / ASSET_NAME):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{ASSET_NAME} not found next to this lesson. It ships in assets/ and is never "
        "downloaded by the lesson; restore it from the lesson directory."
    )


_data = np.load(_asset_path())
STATION_ID = [str(s) for s in _data["station_id"]]
STATION_NAME = [str(s) for s in _data["station_name"]]
AREA_KM2 = [float(a) for a in _data["area_km2"]]
PRECIP = np.asarray(_data["precip_mm_d"], dtype=np.float64)   # (basin, day), mm/day
PET = np.asarray(_data["pet_mm_d"], dtype=np.float64)         # (basin, day), mm/day
FLOW = np.asarray(_data["flow_mm_d"], dtype=np.float64)       # (basin, day), mm/day
N_BASINS, N_DAYS = FLOW.shape
_y, _m, _d = (int(v) for v in _data["start_date"])
DATES = np.datetime64(f"{_y:04d}-{_m:02d}-{_d:02d}") + np.arange(N_DAYS)
DOY = ((DATES - DATES.astype("datetime64[Y]")).astype(int) + 1)   # day of year, 1..366
WARMUP_DAYS = 365   # the model's stores start guessed; the first year is never scored
MONTH = (DATES.astype("datetime64[M]").astype(int) % 12) + 1   # 1..12
WINTER_SHARE = [float(PRECIP[b][(MONTH >= 5) & (MONTH <= 10)].sum() / PRECIP[b].sum())
                for b in range(N_BASINS)]   # share of the rain that falls May to October

print(f"{N_BASINS} catchments · {N_DAYS} daily values each · {DATES[0]} to {DATES[-1]}")
for b in range(N_BASINS):
    ratio = FLOW[b].mean() / PRECIP[b].mean()
    print(f"  {STATION_NAME[b]}: {AREA_KM2[b]:.1f} km2 · mean rain {PRECIP[b].mean():.2f}, "
          f"PET {PET[b].mean():.2f}, flow {FLOW[b].mean():.2f} mm/day · runoff ratio "
          f"{ratio:.2f} · {WINTER_SHARE[b]:.0%} of the rain falls May to October")
print("(May to October is the southern winter half-year: a share far above 50% is a winter-rain "
      "river, far below 50% a summer-rain one)")

## 2. Exercise 1 — `nse()`

The **Nash-Sutcliffe efficiency** scores a simulation against one yardstick: the mean of the
observations. It is 1 for a perfect simulation, 0 for a simulation no better than "always
predict the average flow", and negative for one worse than that (`claims.yaml`:
`knoben-nse-mean-benchmark`). Everything else in this lesson is about what that yardstick hides.

<details><summary>💡 Hint 1 — what to think about</summary>

NSE is one minus a ratio of two sums of squares. The top is how far the simulation is from
the observations. The bottom is how far the *observations* are from their own mean.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Square the simulation error at every day and add it up. Square each observation's distance
from the mean of the OBSERVATIONS (never of the simulation) and add that up. Divide the
first sum by the second and subtract from one. Return a plain Python float.
</details>

In [ ]:
def nse(obs: np.ndarray, sim: np.ndarray) -> float:
    """Nash-Sutcliffe efficiency of `sim` against `obs`: 1 - SSE(sim) / SSE(mean of obs).

    1.0 is perfect, 0.0 matches a constant equal to the mean of `obs`, and the score is
    unbounded below. Returns a float.

    Example:
        >>> obs = np.array([1.0, 2.0, 3.0, 4.0])
        >>> nse(obs, obs)
        1.0
        >>> nse(obs, np.full(4, 2.5))     # the mean of obs, held constant
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _as_float(value: object, name: str) -> float:
    assert isinstance(value, (float, int, np.floating, np.integer)) and not isinstance(value, bool), (
        f"{name} returned {type(value).__name__} {value!r} — return one plain number"
    )
    return float(value)


def _check_nse() -> None:
    obs = np.array([1.0, 2.0, 3.0, 4.0])
    assert abs(_as_float(nse(obs, obs), "nse") - 1.0) < 1e-12, "a perfect simulation must score exactly 1"
    got = _as_float(nse(obs, np.full(4, 2.5)), "nse")
    assert abs(got) < 1e-12, (
        f"predicting the mean of the observations scored {got:.4f}; it must score 0"
    )
    got = _as_float(nse(obs, np.full(4, 3.0)), "nse")
    assert abs(got - (-0.2)) < 1e-12, (
        f"a constant 3.0 against these observations scored {got:.4f}, expected -0.2. The "
        "denominator uses the mean of the OBSERVATIONS, not of the simulation: a biased "
        "simulation must be able to score below zero"
    )
    sim = np.array([1.5, 1.5, 3.5, 3.5])
    got = _as_float(nse(obs, sim), "nse")
    assert abs(got - 0.8) < 1e-12, f"expected 0.8 for a simulation off by 0.5 and 1.5, got {got:.4f}"


_try("exercise 1", _check_nse)

## 3. Exercise 2 — `kge()`

The **Kling-Gupta efficiency** splits the error into three separate questions: do the
fluctuations line up (correlation `r`), is the spread right (`alpha`, the ratio of standard
deviations, simulated over observed), and is the average right (`beta`, the ratio of means,
simulated over observed)? `KGE = 1 - sqrt((r-1)^2 + (alpha-1)^2 + (beta-1)^2)`
(`claims.yaml`: `knoben-kge-components`, `knoben-kge-constant-r`).

One trap: a constant simulation has no correlation with anything. The convention used in the
literature this lesson follows is `r = 0` in that case, so the "always predict the mean"
yardstick has a KGE you can compute. It is not zero.

<details><summary>💡 Hint 1 — what to think about</summary>

Compute the three components first, each on its own, then combine. Check the direction of
each ratio: both put the SIMULATION on top. Compute the correlation yourself from the means
and the deviations, so you control the constant-simulation case.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

If either series is constant (a spread negligible next to its mean, because a constant array's
computed spread can be a rounding error), use `r = 0`. Otherwise `r` is the mean product
of the two deviation series divided by the two population standard deviations. Return the
score together with its three parts, in the order the docstring gives.
</details>

In [ ]:
def kge(obs: np.ndarray, sim: np.ndarray) -> tuple[float, float, float, float]:
    """Kling-Gupta efficiency of `sim` against `obs`. Returns `(kge, r, alpha, beta)`.

    `r` is the Pearson correlation, defined as 0.0 when either series is constant. A series
    counts as constant when its standard deviation is negligible next to its mean (below about
    one part in ten billion of it), because a constant array's computed standard deviation can
    be a rounding error rather than exactly zero; a series that really varies, however little,
    is not constant.
    `alpha = std(sim) / std(obs)` and `beta = mean(sim) / mean(obs)`, and
    `kge = 1 - sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)`.

    Example:
        >>> obs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
        >>> score, r, alpha, beta = kge(obs, 2 * obs + 1)
        >>> round(r, 3), round(alpha, 3), round(beta, 3)
        (1.0, 2.0, 2.333)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_kge() -> None:
    obs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    out = kge(obs, obs)
    assert isinstance(out, tuple) and len(out) == 4, (
        "kge must return the tuple (kge, r, alpha, beta)"
    )
    assert abs(out[0] - 1.0) < 1e-9, f"a perfect simulation must score 1, got {out[0]:.4f}"
    score, r, alpha, beta = kge(obs, 2.0 * obs + 1.0)
    assert abs(r - 1.0) < 1e-9, f"a linear rescaling is perfectly correlated, got r={r:.4f}"
    assert abs(alpha - 2.0) < 1e-9, (
        f"alpha was {alpha:.4f}; it is std(sim) / std(obs), simulated on top, so 2.0 here"
    )
    assert abs(beta - 7.0 / 3.0) < 1e-9, (
        f"beta was {beta:.4f}; it is mean(sim) / mean(obs), simulated on top, so 7/3 here"
    )
    score, r, alpha, beta = kge(obs, np.full(5, obs.mean()))
    assert r == 0.0 and alpha == 0.0 and abs(beta - 1.0) < 1e-12, (
        "a constant simulation has r = 0 by convention, alpha = 0, and beta = 1 if it sits at "
        "the observed mean"
    )
    assert abs(score - (1.0 - math.sqrt(2.0))) < 1e-9, (
        f"the always-predict-the-mean simulation must score 1 - sqrt(2), not {score:.4f}"
    )


_try("exercise 2", _check_kge)

## 4. Exercise 3 — a benchmark that respects the calendar

"Always predict the mean" is a weak yardstick when rivers are seasonal: a forecaster who
only knew the date would already beat it by a mile (`claims.yaml`: `knoben-seasonal-benchmark`). A
fairer yardstick is the **day-of-year climatology**: for each calendar day, the mean flow
on that day in the years you are allowed to look at. Two functions:
`doy_climatology()` builds it, and `benchmark_skill()` scores a simulation *against any
benchmark* instead of against the mean.

<details><summary>💡 Hint 1 — what to think about</summary>

Group the flows by day of year (1 to 366) and average each group. A single day's average
over a few years is noisy, so smooth it with a moving average that wraps round the year
end. `smooth_days` counts days on EACH side of the centre.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sum the flows and count the days for each of the 366 groups, then divide. To smooth, average
each day with its neighbours from `smooth_days` before to `smooth_days` after, treating day 366
and day 1 as neighbours. Skill against a benchmark is one minus the simulation's squared error
divided by the benchmark's squared error, both against the observations.
</details>

In [ ]:
def doy_climatology(doy: np.ndarray, q: np.ndarray, smooth_days: int = 15) -> np.ndarray:
    """Mean of `q` for each day of year, as a length-366 array (index 0 is day of year 1).

    `doy` holds integer days of year (1..366), one per value of `q`, and every day of year
    from 1 to 366 must appear at least once. Then each of the 366 means is replaced by the
    plain average of itself and its `smooth_days` neighbours on each side (a window of
    `2 * smooth_days + 1` values), wrapping from day 366 round to day 1. `smooth_days=0`
    leaves the raw means.

    Example:
        >>> doy = np.tile(np.arange(1, 367), 2)
        >>> q = doy.astype(float)
        >>> clim = doy_climatology(doy, q, smooth_days=0)
        >>> float(clim[0]), float(clim[365])
        (1.0, 366.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def benchmark_skill(obs: np.ndarray, sim: np.ndarray, benchmark: np.ndarray) -> float:
    """Skill of `sim` against `benchmark`: 1 - SSE(sim) / SSE(benchmark), both against `obs`.

    1.0 is perfect, 0.0 means no better than the benchmark, negative means worse. With a
    constant benchmark equal to the mean of `obs` this is exactly the NSE.

    Example:
        >>> obs = np.array([1.0, 2.0, 3.0, 4.0])
        >>> round(benchmark_skill(obs, obs + 0.1, np.full(4, 2.5)), 3)
        0.992
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_benchmarks() -> None:
    doy = np.tile(np.arange(1, 367), 3)
    q = np.concatenate([np.full(366, 1.0), np.full(366, 2.0), np.full(366, 6.0)])
    clim = doy_climatology(doy, q, smooth_days=0)
    assert clim.shape == (366,), f"expected 366 values, got shape {clim.shape}"
    assert np.allclose(clim, 3.0), "the raw mean of 1, 2 and 6 on every day is 3"
    spike = np.zeros(366)
    spike[0] = 366.0
    smooth = doy_climatology(np.arange(1, 367), spike, smooth_days=1)
    assert np.allclose(smooth[[365, 0, 1]], 122.0) and smooth[2] == 0.0, (
        "with smooth_days=1 a spike on day 1 must spread to day 366 AND day 2 (the year wraps) "
        "and stop there: the window is 2 * smooth_days + 1 values wide"
    )
    obs = np.array([1.0, 2.0, 3.0, 4.0])
    sim = np.array([1.5, 1.5, 3.5, 3.5])
    against_mean = 1.0 - np.sum((obs - sim) ** 2) / np.sum((obs - obs.mean()) ** 2)
    assert abs(benchmark_skill(obs, sim, np.full(4, obs.mean())) - against_mean) < 1e-12, (
        "skill against a constant equal to the observed mean is exactly the NSE"
    )
    assert abs(benchmark_skill(obs, obs, np.full(4, 0.0)) - 1.0) < 1e-12, "a perfect simulation has skill 1"
    assert abs(benchmark_skill(obs, sim, sim)) < 1e-12, "a simulation identical to the benchmark has skill 0"
    assert benchmark_skill(obs, np.full(4, 0.0), np.full(4, 2.5)) < 0.0, (
        "a simulation worse than the benchmark must have negative skill"
    )


_try("exercise 3", _check_benchmarks)

Two demos on the real records, each computed by your functions. First, what the "always
predict the mean" yardstick scores under both metrics. Second, what a forecaster who knew
nothing but the date would score: the day-of-year climatology of the whole record (in-sample,
the most generous form of this benchmark).

In [ ]:
RESULTS: dict[int, dict[str, float]] = {b: {} for b in range(N_BASINS)}


def _z(x: float) -> float:
    """Print-safe zero: never show a negative zero."""
    return 0.0 if abs(x) < 5e-13 else float(x)


def _demo_yardsticks() -> None:
    for b in range(N_BASINS):
        obs = FLOW[b][WARMUP_DAYS:]
        const = np.full(len(obs), obs.mean())
        score, r, alpha, beta = kge(obs, const)
        print(f"{STATION_ID[b]}: always predict the mean flow -> NSE {_z(nse(obs, const)):+.3f}, "
              f"KGE {score:+.3f} (r {r:.1f}, alpha {alpha:.1f}, beta {beta:.1f})")
    print(f"so the mean-flow yardstick sits at KGE = 1 - sqrt(2) = {1 - math.sqrt(2):+.3f}, "
          "not at 0 (claims.yaml: knoben-kge-mean-flow)")


def _demo_calendar() -> None:
    scores = []
    for b in range(N_BASINS):
        obs, doy = FLOW[b][WARMUP_DAYS:], DOY[WARMUP_DAYS:]
        calendar = doy_climatology(doy, obs)[doy - 1]
        RESULTS[b]["calendar_in_sample_nse"] = nse(obs, calendar)
        scores.append(nse(obs, calendar))
        print(f"{STATION_ID[b]} ({WINTER_SHARE[b]:.0%} of its rain May to October): the date alone scores "
              f"NSE {nse(obs, calendar):+.3f}, KGE {kge(obs, calendar)[0]:+.3f}")
    print(f"so an NSE of, say, 0.5 clears the calendar by a margin that depends on the river: the "
          f"calendar alone already scores {max(scores):+.3f} in one of them and {min(scores):+.3f} in the other")


_try("yardsticks demo", _demo_yardsticks, needs=("exercise 1", "exercise 2"))
_try("calendar demo", _demo_calendar, needs=("exercise 1", "exercise 2", "exercise 3"))

## 5. The model, and what it leaves out

The model is a four-parameter, daily, **lumped** conceptual model in the GR4J family
(Perrin, Michel and Andréassian, 2003; the equations below were checked against the
reference implementation in the airGR package and against the model's open description by its
first author, `claims.yaml`: `gr4j-reference-citation`, `gr4j-*`). It is a
**simplification, and this lesson's own code is not that package**: no snow, a fixed way of
starting the stores, and one simple search to fit four numbers. What it keeps is the shape of
the idea.

Rain first meets a **production store** (capacity `X1`, mm): net rain fills it, PET drains it,
and some water percolates out. What leaves it is split **90 / 10**; the 90 % goes through
one **unit hydrograph** (a fixed delay curve of time base `X4` days) into a **routing store**
(capacity `X3`, mm), which drains slowly, and the 10 % goes through a second, longer unit
hydrograph straight to the river. `X2` (mm/day) is a groundwater exchange term. Total flow
is the routing store's outflow plus the direct branch. **The routing store is the model's
short-term memory**: it spreads a storm's water over the days and weeks after the rain stopped.
The production store remembers longer still: it keeps percolating a little water for months.

In [ ]:
# The four parameters, a plain starting guess, and the range the search may roam. These are this
# lesson's own choices. X1 and X3 are searched on a log scale (a store of 100 mm and one of
# 1000 mm differ by a factor, not by 900), and the search space is deliberately wide.
PARAM_NAMES = ("X1 production store mm", "X2 exchange mm/day", "X3 routing store mm",
               "X4 unit hydrograph days")
DEFAULT_PARAMS = (350.0, 0.0, 90.0, 1.7)
_Z_LOWER = np.array([math.log(1.0), -5.0, math.log(1.0), 0.5])
_Z_UPPER = np.array([math.log(2500.0), 3.0, math.log(1000.0), 4.0])
_Z_STEP = np.array([0.5, 0.5, 0.5, 0.3])


def params_to_z(params: tuple[float, float, float, float]) -> np.ndarray:
    """Parameters -> the search coordinates (X1 and X3 on a log scale)."""
    return np.array([math.log(params[0]), params[1], math.log(params[2]), params[3]])


def z_to_params(z: np.ndarray) -> tuple[float, float, float, float]:
    """Search coordinates -> parameters."""
    return (math.exp(z[0]), float(z[1]), math.exp(z[2]), float(z[3]))


print("parameters:", "; ".join(PARAM_NAMES))
print("starting guess:", DEFAULT_PARAMS)

## 6. Exercise 4 — `gr4j_unit_hydrographs()`

A unit hydrograph says how one day's runoff is spread over the following days. Both are built
from a cumulative "S-curve" of time base `c`, with exponent 2.5:

- `SH1(t) = 0` for `t <= 0`, `(t/c)^2.5` for `0 < t < c`, and `1` for `t >= c`;
- `SH2(t) = 0` for `t <= 0`, `0.5 (t/c)^2.5` up to `t = c`, `1 - 0.5 (2 - t/c)^2.5` for
  `c < t < 2c`, and `1` from `2c` on.

The ordinate for day `i` is the S-curve at `i` minus the S-curve at `i - 1`. `UH1` has
`ceil(X4)` ordinates and `UH2` has `ceil(2 * X4)`. Each sums to 1: no water is created or lost.

<details><summary>💡 Hint 1 — what to think about</summary>

Write the two S-curves as small functions of `t` first, exactly as defined above (mind which
comparisons are strict). Then each ordinate is a difference of two S-curve values, one day
apart, for day 1 up to the last day of that hydrograph.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Take `n1 = ceil(X4)` and `n2 = ceil(2 * X4)`. For days `1..n1` compute `SH1(i) - SH1(i-1)`,
and for days `1..n2` compute `SH2(i) - SH2(i-1)`. Return the two arrays, UH1 first. Test your
result by checking that each array sums to one.
</details>

In [ ]:
def gr4j_unit_hydrographs(x4: float) -> tuple[np.ndarray, np.ndarray]:
    """The two unit-hydrograph ordinate arrays `(uh1, uh2)` for time base `x4` (days).

    `uh1` has `ceil(x4)` entries and `uh2` has `ceil(2 * x4)`. Entry `i - 1` of each is the
    S-curve at day `i` minus the S-curve at day `i - 1` (definitions in the text above).

    Example:
        >>> uh1, uh2 = gr4j_unit_hydrographs(1.0)
        >>> [float(v) for v in uh1], [float(v) for v in uh2]
        ([1.0], [0.5, 0.5])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_unit_hydrographs() -> None:
    for x4 in (0.6, 1.0, 1.7, 2.0, 3.3):
        uh1, uh2 = gr4j_unit_hydrographs(x4)
        assert len(uh1) == math.ceil(x4) and len(uh2) == math.ceil(2 * x4), (
            f"for X4={x4} expected {math.ceil(x4)} and {math.ceil(2 * x4)} ordinates, got "
            f"{len(uh1)} and {len(uh2)} — UH1 has ceil(X4), UH2 has ceil(2 * X4)"
        )
        assert abs(uh1.sum() - 1.0) < 1e-12 and abs(uh2.sum() - 1.0) < 1e-12, (
            f"the ordinates for X4={x4} sum to {uh1.sum():.6f} and {uh2.sum():.6f}; each "
            "hydrograph must sum to exactly 1 or the model creates or destroys water"
        )
        assert np.all(uh1 >= 0) and np.all(uh2 >= 0), "ordinates cannot be negative"
    grid = [2.19, 3.2] + [round(0.5 + 0.01 * i, 2) for i in range(351)] + [1.7 + 0.3 * k for k in range(-4, 8)]
    with _time_limit(2, "gr4j_unit_hydrographs has not returned after 2 s. A loop that runs until the "
                     "ordinates add up to 1 can spin for ever, because rounding can leave the running sum "
                     "a hair below 1: loop over a fixed number of days, ceil(X4) and ceil(2 * X4)"):
        for x4 in grid:
            uh1, uh2 = gr4j_unit_hydrographs(x4)
            assert len(uh1) == math.ceil(x4) and len(uh2) == math.ceil(2 * x4), (
                f"X4={x4}: expected {math.ceil(x4)} and {math.ceil(2 * x4)} ordinates, got {len(uh1)} and {len(uh2)}"
            )
    uh1, uh2 = gr4j_unit_hydrographs(2.0)
    assert abs(uh1[0] - 0.5 ** 2.5) < 1e-12, (
        "the first UH1 ordinate for X4=2 is the S-curve at day 1: (1/2)^2.5"
    )
    assert abs(uh2[0] - 0.5 * 0.5 ** 2.5) < 1e-12, (
        "the first UH2 ordinate for X4=2 is SH2 at day 1: 0.5 * (1/2)^2.5"
    )


_try("exercise 4", _check_unit_hydrographs)

## 7. Exercise 5 — `gr4j_simulate()`

One day at a time, with `S` the production store and `R` the routing store, starting from
`S = 0.3 * X1` and `R = 0.5 * X3` (the first year is never scored, so the guess is forgotten).
For each day with rain `P` and PET `E`:

1. If `P >= E`: net rain `Pn = P - E`, and `Ps = X1 (1 - (S/X1)^2) tanh(Pn/X1) /
   (1 + (S/X1) tanh(Pn/X1))` fills the store, `S += Ps`; the rest `Pn - Ps` runs off.
   Otherwise `En = E - P`, the store loses `Es = S (2 - S/X1) tanh(En/X1) / (1 + (1 - S/X1)
   tanh(En/X1))`, and nothing runs off.
2. Percolation `Perc = S (1 - (1 + (S/(2.25 X1))^4)^(-1/4))` leaves the store: `S -= Perc`.
   Water leaving to the rivers is `Pr = (Pn - Ps) + Perc`.
3. `0.9 Pr` enters UH1 and `0.1 Pr` enters UH2 (each day's input is spread over the following
   days by the ordinates). `Q9` and `Q1` are what the two hydrographs release *today*.
4. Exchange `F = X2 (R/X3)^3.5`. `R = max(0, R + Q9 + F)`, then `Qr = R (1 - (1 + (R/X3)^4)^(-1/4))`
   leaves it: `R -= Qr`. The direct branch gives `Qd = max(0, Q1 + F)`. Flow is `Qr + Qd`.

<details><summary>💡 Hint 1 — what to think about</summary>

Keep the two stores as plain numbers, and each unit hydrograph as a short list of water that
is still "on its way": every day, today's input is added along that list scaled by the
ordinates, then the first entry is released and the rest shift one day closer. Work in the
order of the numbered steps; several formulas need the store as it is at that moment.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the ordinates from your `gr4j_unit_hydrographs`. Loop over the days with plain floats and
`math.tanh`: pure Python is fast enough for a few thousand days. Watch the two branches of
step 1 (`P >= E` or not), and remember the routing store's outflow and the direct branch are
added at the end. Collect one flow per day into a numpy array as long as `precip`.
</details>

In [ ]:
def gr4j_simulate(precip: np.ndarray, pet: np.ndarray,
                  params: tuple[float, float, float, float]) -> np.ndarray:
    """Daily flow (mm/day) from rain and PET (mm/day), as described in steps 1-4 above.

    `params = (X1, X2, X3, X4)`. The stores start at `0.3 * X1` and `0.5 * X3`. Returns an
    array as long as `precip`, never negative. Does not modify its inputs.

    Example:
        >>> q = gr4j_simulate(np.zeros(5), np.zeros(5), (350.0, 0.0, 90.0, 1.7))
        >>> len(q), bool(np.all(q >= 0)), bool(q[0] > 0)   # the routing store starts part-full
        (5, True, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_gr4j_simulate() -> None:
    params = (350.0, 0.0, 90.0, 1.7)
    rain = np.zeros(120)
    rain[10] = 40.0
    q = gr4j_simulate(rain, np.zeros(120), params)
    assert isinstance(q, np.ndarray) and q.shape == (120,), "return one flow per day, as a numpy array"
    assert np.all(q >= 0) and np.all(np.isfinite(q)), "flow must be finite and never negative"
    base = gr4j_simulate(np.zeros(120), np.zeros(120), params)
    storm = q - base
    assert storm[10:13].max() > 0.5 and abs(storm[:10]).max() < 1e-9, (
        "the storm on day 10 must raise flow from day 10 on and not before it"
    )
    assert storm[60:].sum() > 0.2 * storm.sum(), (
        "after 50 dry days the model has released almost none of the storm's water. The slow "
        "tail comes from the ROUTING store: check that 0.9 of Pr goes through UH1 into the "
        "routing store (R) and that R drains by Qr each day, instead of passing straight to the river"
    )
    q2 = gr4j_simulate(rain, np.zeros(120), params)
    assert np.array_equal(q, q2), "the same inputs must give the same flow: no hidden state between calls"
    dry = gr4j_simulate(np.zeros(60), np.full(60, 3.0), params)
    assert dry[-1] < dry[0], "with no rain the flow must recede"
    wet = gr4j_simulate(np.full(80, 12.0), np.full(80, 1.0), params)
    assert wet[-1] > 5.0 and wet[-1] < 12.0, (
        f"steady rain of 12 with PET 1 should settle to a flow below the rain, got {wet[-1]:.2f}"
    )


_try("exercise 5", _check_gr4j_simulate, needs=("exercise 4",))

Before fitting anything, look at the routing store at work. Put one 40 mm storm into the
model on an otherwise dry record (no rain, no PET), subtract the flow the model produces with
no storm at all, and see how much of the storm's water is still on its way after 5, 20 and 60
days. Then do it again with a routing store of 1 mm, too small to hold anything back. Where the
two lines differ is the routing store's doing; what both still have to deliver after two months
is the production store's slow percolation.

In [ ]:
def _demo_routing_tail() -> None:
    days, later = 200, (5, 20, 60)
    dry, storm = np.zeros(days), np.zeros(days)
    storm[10] = 40.0
    still_to_come = {}
    for x3 in (DEFAULT_PARAMS[2], 1.0):
        params = (DEFAULT_PARAMS[0], DEFAULT_PARAMS[1], x3, DEFAULT_PARAMS[3])
        extra = gr4j_simulate(storm, dry, params) - gr4j_simulate(dry, dry, params)
        total = extra.sum()
        still_to_come[x3] = [extra[10 + k:].sum() / total for k in later]
        shares = "; ".join(f"after {k} days {s:.0%}" for k, s in zip(later, still_to_come[x3]))
        print(f"routing store {x3:g} mm: {total:.1f} mm of the 40 mm storm reaches the river within "
              f"{days} days, peaking on day {int(np.argmax(extra)) - 10} after the storm; still to come: {shares}")
    gaps = ", ".join(f"after {k} days {100 * (a - b):+.0f}" for k, a, b in
                     zip(later, still_to_come[DEFAULT_PARAMS[2]], still_to_come[1.0]))
    print(f"percentage points more still to come with the {DEFAULT_PARAMS[2]:g} mm routing store than "
          f"with the 1 mm one: {gaps}")


_try("routing demo", _demo_routing_tail, needs=("exercise 4", "exercise 5"))

## 8. Exercise 6 — `pattern_search()`: a calibration you can run twice

Calibration means choosing the four numbers so the simulation matches the record. The
simplest honest search is a **compass (pattern) search**: nudge one coordinate at a time by a
step, keep any nudge that improves the score, and when a whole sweep improves nothing, halve
the steps. It is deterministic (same start, same answer), needs no gradients and is short.

<details><summary>💡 Hint 1 — what to think about</summary>

Keep a current point and its score. One *sweep* visits every coordinate; for each you try
one step up and, if that did not help, one step down. A trial that lands outside the bounds
is pulled back to the bound; if that leaves the point unchanged, do not spend an evaluation
on it. Count every call to `f`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Evaluate `f` at the start (that is evaluation one). Repeat while the largest step is at least
`min_step` and evaluations remain: sweep through the coordinates, accepting a trial the
moment it is strictly lower than the current score; if a full sweep accepted nothing, halve
every step. Return the point, its score and the number of evaluations used.
</details>

In [ ]:
def pattern_search(f: Callable[[np.ndarray], float], x0: np.ndarray, lower: np.ndarray,
                   upper: np.ndarray, step: np.ndarray, min_step: float,
                   max_evals: int) -> tuple[np.ndarray, float, int]:
    """Minimise `f` by compass search inside the box `lower <= x <= upper`.

    Start at `x0` (one evaluation). A *sweep* takes each coordinate in turn, tries `+step[i]`
    then `-step[i]`, clips the trial into the box, skips it if clipping leaves the coordinate
    unchanged, evaluates `f`, and moves to the trial the moment its value is strictly lower.
    If a whole sweep moved nowhere, all steps are halved. Stop when every step is below
    `min_step`, or when `max_evals` evaluations have been used (never exceed it).
    Returns `(x_best, f_best, n_evals)`; `n_evals` counts every call to `f`.

    Example:
        >>> f = lambda x: float((x[0] - 2.0) ** 2 + (x[1] + 1.0) ** 2)
        >>> x, fx, n = pattern_search(f, np.array([0.0, 0.0]), np.array([-5.0, -5.0]),
        ...                           np.array([5.0, 5.0]), np.array([1.0, 1.0]), 1e-3, 500)
        >>> bool(abs(x[0] - 2.0) < 0.01 and abs(x[1] + 1.0) < 0.01), n <= 500
        (True, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pattern_search() -> None:
    calls: list[np.ndarray] = []

    def bowl(x: np.ndarray) -> float:
        calls.append(np.array(x, dtype=float))
        assert len(calls) <= 5000, (
            "f has been called more than 5000 times in one search, so the search never stops: halve "
            "the steps after a sweep that moved nowhere, and count every call against max_evals"
        )
        return float((x[0] - 0.3) ** 2 + 4.0 * (x[1] - 1.7) ** 2)

    lower, upper = np.array([-2.0, -2.0]), np.array([2.0, 2.0])
    x, fx, n = pattern_search(bowl, np.array([-1.0, -1.0]), lower, upper,
                              np.array([0.5, 0.5]), 0.005, 400)
    assert abs(x[0] - 0.3) < 0.02 and abs(x[1] - 1.7) < 0.02, (
        f"ended at {np.round(x, 3)}, expected near (0.3, 1.7): the minimum lies BETWEEN "
        "grid points, so the steps must be halved when a sweep improves nothing"
    )
    assert n == len(calls), f"reported {n} evaluations but f was called {len(calls)} times"
    assert abs(fx - bowl(x)) < 1e-12, "the returned score must be f at the returned point"
    assert all(np.all(c >= lower) and np.all(c <= upper) for c in calls), (
        "f was called outside the bounds: clip each trial into [lower, upper]"
    )
    calls.clear()
    x, fx, n = pattern_search(bowl, np.array([0.0, 0.0]), lower, np.array([2.0, 1.0]),
                              np.array([0.5, 0.5]), 0.005, 400)
    assert x[1] <= 1.0 and abs(x[1] - 1.0) < 0.01, (
        "the unconstrained minimum is above the upper bound of 1.0, so the search must end at that bound"
    )
    calls.clear()
    x, fx, n = pattern_search(bowl, np.array([-1.0, -1.0]), lower, upper,
                              np.array([0.5, 0.5]), 1e-9, 12)
    assert n <= 12 and len(calls) <= 12, (
        f"max_evals=12 was exceeded ({len(calls)} calls to f): stop the moment the budget is spent"
    )
    x0 = np.array([0.3, 1.7])
    x, fx, n = pattern_search(bowl, x0, lower, upper, np.array([0.5, 0.5]), 0.01, 400)
    assert np.allclose(x, x0) and fx == 0.0, "starting at the exact minimum, the search must stay there"


_try("exercise 6", _check_pattern_search)

The next cell is given, not an exercise: `calibrate_gr4j()` wires your search to your model.
It fits the four parameters to the days you hand it (never looking at any others) by
maximising a score you choose, NSE unless told otherwise. Two details matter later. The
first year is never scored, and the simulation only runs as far as the last day it is fitted
on, so a model fitted to early years has literally not seen the later rain.

In [ ]:
def calibrate_gr4j(precip: np.ndarray, pet: np.ndarray, obs: np.ndarray, fit_idx: np.ndarray,
                   objective: Callable[[np.ndarray, np.ndarray], float] | None = None,
                   max_evals: int = 400) -> tuple[float, float, float, float]:
    """Fit (X1, X2, X3, X4) by maximising `objective(obs[fit], sim[fit])` over the days `fit_idx`."""
    objective = nse if objective is None else objective
    fit = np.asarray(fit_idx)
    fit = fit[fit >= WARMUP_DAYS]
    last = int(fit.max()) + 1

    def loss(z: np.ndarray) -> float:
        sim = gr4j_simulate(precip[:last], pet[:last], z_to_params(z))
        score = float(objective(obs[fit], sim[fit]))
        return -score if math.isfinite(score) else 1.0e6

    z, _, _ = pattern_search(loss, params_to_z(DEFAULT_PARAMS), _Z_LOWER, _Z_UPPER, _Z_STEP,
                             0.05, max_evals)
    return z_to_params(z)


def simulate_gr4j(precip: np.ndarray, pet: np.ndarray,
                  params: tuple[float, float, float, float]) -> np.ndarray:
    """The whole record, run continuously from its first day (given: it just calls yours)."""
    return gr4j_simulate(precip, pet, params)

## 9. Exercise 7 — an honest hold-out

A score means something only on days the model was not fitted to. Two functions make that
hard to get wrong. `chronological_split()` cuts the record in TIME: the past trains, the
future tests, optionally with a **gap** so the last training days do not neighbour the first
test days. `holdout_evaluation()` then fits on the training days only and scores twice: on
the days it was fitted to (in-sample) and on the days it never saw.

<details><summary>💡 Hint 1 — what to think about</summary>

The test set is the LAST `round(n * test_frac)` days. With a gap of `g`, the `g` days just
before the test set belong to neither set. For the hold-out: which days may `calibrate` see?
And the model must run once over the whole record, because the stores carry memory forward
from day to day; only the *scoring* is restricted to a set of days.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Split: the test indices are the final block; the training indices are the first block, ending
`gap` days before the test block starts. Both are integer index arrays in increasing order.
Hold-out: call `calibrate` with the TRAIN indices only, call `simulate` once on the full
arrays with the fitted parameters, then apply `metric` to the observed and simulated values
at the train days, and again at the test days. Return `(train_score, test_score)`.
</details>

In [ ]:
def chronological_split(n: int, test_frac: float, gap: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Split day indices `0..n-1` by time. Returns `(train_idx, test_idx)`, both increasing.

    The test set is the last `n_test = round(n * test_frac)` days. `gap` days immediately
    before it are dropped from both sets. The training set is everything before that.

    Example:
        >>> train, test = chronological_split(10, 0.3, gap=1)
        >>> [int(v) for v in train], [int(v) for v in test]
        ([0, 1, 2, 3, 4, 5], [7, 8, 9])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def holdout_evaluation(simulate: Callable, calibrate: Callable, metric: Callable,
                       precip: np.ndarray, pet: np.ndarray, obs: np.ndarray,
                       train_idx: np.ndarray, test_idx: np.ndarray) -> tuple[float, float]:
    """Fit on `train_idx` only, then score in-sample and out-of-sample. Returns `(train, test)`.

    `calibrate(precip, pet, obs, fit_idx)` returns fitted parameters using only the days in
    `fit_idx`. `simulate(precip, pet, params)` is run ONCE on the full arrays. Then
    `metric(obs[idx], sim[idx])` is applied to `train_idx` and to `test_idx`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_holdout() -> None:
    train, test = chronological_split(100, 0.25)
    assert len(test) == 25 and len(train) == 75, f"expected 75 train and 25 test days, got {len(train)} and {len(test)}"
    assert train.max() < test.min() and np.array_equal(test, np.arange(75, 100)), (
        "the test days must be the LAST days, entirely after every training day. A shuffled "
        "or random selection leaks the future into training"
    )
    train, test = chronological_split(100, 0.25, gap=5)
    assert len(train) == 70 and train.max() == 69 and test.min() == 75, (
        f"with gap=5 the days 70..74 belong to neither set; got train up to {train.max()}, test from {test.min()}"
    )
    seen: dict[str, object] = {}

    def fake_calibrate(p, e, o, idx):
        seen["fit"] = np.asarray(idx)
        return (1.0,)

    def fake_simulate(p, e, prm):
        seen["n"] = len(p)
        return np.asarray(p, dtype=float) * prm[0]

    obs = np.arange(20.0)
    train, test = np.arange(0, 12), np.arange(14, 20)
    tr, te = holdout_evaluation(fake_simulate, fake_calibrate, lambda o, s: float(np.mean(s - o)),
                                np.arange(20.0) + 1.0, np.zeros(20), obs, train, test)
    assert np.array_equal(seen["fit"], train), (
        "calibrate must be given the TRAINING days and nothing else: fitting on the test days "
        "(or on all of them) makes every score optimistic"
    )
    assert seen["n"] == 20, "run the simulation once over the whole record, then pick out the days to score"
    assert abs(tr - 1.0) < 1e-12 and abs(te - 1.0) < 1e-12, "the metric compares obs with sim at the chosen days"
    tr, te = holdout_evaluation(fake_simulate, fake_calibrate, lambda o, s: float(np.sum(o)),
                                np.arange(20.0), np.zeros(20), obs, train, test)
    assert tr == float(obs[train].sum()) and te == float(obs[test].sum()), (
        "return (train score, test score) in that order, each on its own days"
    )


_try("exercise 7", _check_holdout)

### Fit it, then score it on years it never saw

The record is split in time: the first half fits, the second half tests. `holdout_evaluation`
fits the four parameters to the first half with your search, then reports two NSE scores: on
the years it was fitted to, and on the later years it never saw. The same catchments also tell
you whether the two halves are alike: compare their mean rain and mean flow.

In [ ]:
_FITS: dict[tuple[str, int], dict] = {}
TEST_FRAC = 0.5


def _chrono_fit(b: int) -> dict:
    """Fit once per catchment on the first half of the record (cached, so every demo shares it)."""
    key = ("chrono", b)
    if key not in _FITS:
        train_idx, test_idx = chronological_split(N_DAYS, TEST_FRAC)
        train_idx = train_idx[train_idx >= WARMUP_DAYS]
        fitted: dict[str, object] = {}

        def calibrate(p, e, o, idx):
            fitted["params"] = calibrate_gr4j(p, e, o, idx)
            return fitted["params"]

        train_score, test_score = holdout_evaluation(simulate_gr4j, calibrate, nse, PRECIP[b],
                                                     PET[b], FLOW[b], train_idx, test_idx)
        params = fitted["params"]
        _FITS[key] = dict(train_idx=train_idx, test_idx=test_idx, params=params,
                          sim=simulate_gr4j(PRECIP[b], PET[b], params),
                          train_nse=train_score, test_nse=test_score)
    return _FITS[key]


def _demo_holdout() -> None:
    for b in range(N_BASINS):
        fit = _chrono_fit(b)
        tr, te = fit["train_idx"], fit["test_idx"]
        if b == 0:
            print(f"fitted on {DATES[tr[0]]} to {DATES[tr[-1]]}; tested on {DATES[te[0]]} to {DATES[te[-1]]}")
        obs, sim = FLOW[b], fit["sim"]
        calendar = doy_climatology(DOY[tr], obs[tr])[DOY[te] - 1]
        fit["calendar_test_nse"] = nse(obs[te], calendar)
        fit["skill_vs_calendar"] = benchmark_skill(obs[te], sim[te], calendar)
        fit["test_kge"] = kge(obs[te], sim[te])[0]
        RESULTS[b].update(train_nse=fit["train_nse"], test_nse=fit["test_nse"],
                          calendar_test_nse=fit["calendar_test_nse"],
                          skill_vs_calendar=fit["skill_vs_calendar"], test_kge=fit["test_kge"])
        shown = ", ".join(f"{v:.1f}" for v in fit["params"])
        print(f"{STATION_ID[b]}: X1..X4 = {shown}")
        print(f"   NSE on the fitted years {fit['train_nse']:+.3f} · on the unseen years "
              f"{fit['test_nse']:+.3f} · KGE on the unseen years {fit['test_kge']:+.3f}")
        print(f"   the date alone, on the unseen years: NSE {fit['calendar_test_nse']:+.3f}; "
              f"skill of the model against that calendar: {fit['skill_vs_calendar']:+.3f}")
        d_rain = PRECIP[b][te].mean() / PRECIP[b][tr].mean() - 1
        d_flow = FLOW[b][te].mean() / FLOW[b][tr].mean() - 1
        print(f"   mean rain changed {d_rain:+.0%} from the fitted years to the unseen years; "
              f"mean flow changed {d_flow:+.0%}")


_try("hold-out demo", _demo_holdout, needs=tuple(f"exercise {i}" for i in range(1, 8)))

Scoring on the fitted years is the flattering number. The gap between the two NSE values above
is measured, not assumed: compare it between the two rivers. The last line of each block shows
how much the later years differ from the earlier ones in mean rain and mean flow; the more they
differ, the more an honest hold-out matters, because it is the only score that tells you what
a forecast made *today* is worth.

### The split that looks harmless: random days

`random_split` (given) shuffles the days with a fixed integer hash, so it is identical on every
machine. Fitting on a random half of the days and scoring on the other half feels
fair: no day is in both. But a river's flow today is almost its flow yesterday, and the
weather regime of a wet year is shared by its days; the demo measures how much that matters.
To show the leak without any model at all,
`nearest_day_interpolation` (given) predicts each held-out day by drawing a straight line
between the nearest fitted days. It never looks at rain.

In [ ]:
def random_split(n: int, test_frac: float, seed: int = 2026,
                 start: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """A seeded random split of the days `start..n-1` into `(train_idx, test_idx)`, both sorted.

    The shuffle is a fixed integer hash rather than numpy's generator, so it gives the same
    split on every machine and every numpy version.
    """
    days = np.arange(start, n, dtype=np.uint64)
    x = days + np.uint64(seed)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xFF51AFD7ED558CCD)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xC4CEB9FE1A85EC53)
    x ^= x >> np.uint64(33)
    shuffled = days[np.argsort(x, kind="stable")].astype(int)
    n_test = int(round((n - start) * test_frac))
    return np.sort(shuffled[n_test:]), np.sort(shuffled[:n_test])


def nearest_day_interpolation(train_idx: np.ndarray, obs_train: np.ndarray,
                              test_idx: np.ndarray) -> np.ndarray:
    """A 'model' that uses no rain at all: draw straight lines between fitted days' flows."""
    return np.interp(test_idx, train_idx, obs_train)


def _demo_random_split() -> None:
    for b in range(N_BASINS):
        chrono = _chrono_fit(b)
        train_r, test_r = random_split(N_DAYS, TEST_FRAC, start=WARMUP_DAYS)
        fitted: dict[str, object] = {}

        def calibrate(p, e, o, idx):
            fitted["params"] = calibrate_gr4j(p, e, o, idx)
            return fitted["params"]

        tr_r, te_r = holdout_evaluation(simulate_gr4j, calibrate, nse, PRECIP[b], PET[b], FLOW[b],
                                        train_r, test_r)
        sim_r = simulate_gr4j(PRECIP[b], PET[b], fitted["params"])
        te_c = nse(FLOW[b][chrono["test_idx"]], sim_r[chrono["test_idx"]])
        interp_random = nse(FLOW[b][test_r], nearest_day_interpolation(train_r, FLOW[b][train_r], test_r))
        interp_chrono = nse(FLOW[b][chrono["test_idx"]], nearest_day_interpolation(
            chrono["train_idx"], FLOW[b][chrono["train_idx"]], chrono["test_idx"]))
        RESULTS[b].update(random_test_nse=te_r, random_params_on_chrono_test_nse=te_c,
                          interp_random_nse=interp_random, interp_chrono_nse=interp_chrono)
        print(f"{STATION_ID[b]}: model fitted on random days, scored on the other random days: NSE "
              f"{te_r:+.3f}   (chronological hold-out: {chrono['test_nse']:+.3f})")
        print(f"   the same random-fitted parameters on the later years: NSE {te_c:+.3f}; those "
              "years were already inside their fit")
        print(f"   rain-free interpolation between fitted days: random split NSE {interp_random:+.3f}, "
              f"chronological split NSE {interp_chrono:+.3f}")


_try("random split demo", _demo_random_split, needs=tuple(f"exercise {i}" for i in range(1, 8)))

## 10. Exercise 8 — a data-driven baseline: regression on lagged rain

Is the conceptual model earning its keep? The fair competitor knows the same rain and nothing
else: a **ridge regression** from the last few weeks of rainfall to today's flow. Two
functions. `lagged_matrix()` turns a rain series into a table whose column `k` is the rain
`k` days ago (missing days before the record starts count as no rain). `ridge_fit()` fits
a straight-line model with an intercept, and shrinks the coefficients, but not the intercept.

<details><summary>💡 Hint 1 — what to think about</summary>

Lag 0 is today, lag 1 is yesterday. Only the past may appear in a row: a column that shifts
the series the wrong way, or one that wraps the end of the record round to the start, hands
the model rain it could not have known. For the fit, add a column of ones so the intercept is
just another coefficient, then decide which coefficient the penalty must leave alone.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Lag matrix: allocate zeros, then for each lag `k` copy the series shifted `k` days later into
column `k` (the first `k` rows stay zero). Ridge: the coefficients solve the normal equations
with `lam` added to the diagonal of `A^T A`, where `A` has the ones column first; set the
penalty for the intercept's diagonal entry to zero. Use `np.linalg.solve`. Return intercept
first, then one weight per column.
</details>

In [ ]:
def lagged_matrix(x: np.ndarray, n_lags: int) -> np.ndarray:
    """Table with `len(x)` rows and `n_lags` columns; column `k` holds `x` from `k` days earlier.

    Row `t` is `[x[t], x[t-1], ..., x[t-n_lags+1]]`. Days before the start of `x` are 0.0.
    Never uses a value from a later day, and does not wrap round.

    Example:
        >>> [[float(v) for v in row] for row in lagged_matrix(np.array([1.0, 2.0, 3.0]), 2)]
        [[1.0, 0.0], [2.0, 1.0], [3.0, 2.0]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def ridge_fit(X: np.ndarray, y: np.ndarray, lam: float) -> np.ndarray:
    """Ridge regression with an intercept that is NOT penalised. Returns `[intercept, *weights]`.

    Minimises `sum((y - intercept - X @ w)**2) + lam * sum(w**2)`. Returns an array of length
    `X.shape[1] + 1`, intercept first.

    Example:
        >>> X = np.array([[0.0], [1.0], [2.0], [3.0]])
        >>> w = ridge_fit(X, 5.0 + 2.0 * X[:, 0], lam=0.0)
        >>> [round(float(v), 6) for v in w]
        [5.0, 2.0]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def ridge_predict(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Predictions from `ridge_fit`'s weights (given). Flow cannot be negative, so clip at 0."""
    return np.maximum(w[0] + np.asarray(X, dtype=float) @ w[1:], 0.0)


def _check_baseline() -> None:
    lagged = lagged_matrix(np.array([1.0, 2.0, 3.0, 4.0]), 3)
    expect = np.array([[1.0, 0.0, 0.0], [2.0, 1.0, 0.0], [3.0, 2.0, 1.0], [4.0, 3.0, 2.0]])
    assert lagged.shape == (4, 3), f"expected shape (4, 3), got {lagged.shape}"
    assert np.array_equal(lagged, expect), (
        "column k must hold the rain from k days EARLIER, with zeros before the record starts. "
        "A column that wraps the end of the record round to the start (np.roll) leaks the future "
        "into the past"
    )
    rng = np.random.default_rng(7)
    X = rng.normal(0, 1, (200, 3))
    y = 4.0 + X @ np.array([1.0, -2.0, 0.5])
    w = ridge_fit(X, y, 0.0)
    assert w.shape == (4,) and np.allclose(w, [4.0, 1.0, -2.0, 0.5], atol=1e-8), (
        f"expected [intercept, w1, w2, w3] = [4, 1, -2, 0.5] on noiseless data, got {np.round(w, 4)}"
    )
    w = ridge_fit(X, y + 100.0, 1.0e6)
    assert abs(w[0] - 104.0) < 2.0, (
        f"with a huge penalty the intercept should stay near the mean of y (about 104), got {w[0]:.1f}: "
        "the penalty must not be applied to the intercept"
    )
    small, big = ridge_fit(X, y, 1.0), ridge_fit(X, y, 500.0)
    assert np.linalg.norm(big[1:]) < np.linalg.norm(small[1:]), "a larger lam must shrink the weights"


_try("exercise 8", _check_baseline)

Now the fair fight, on the later years neither model saw. Each catchment gets three
forecasters, all fitted only on the first half: the **calendar** (day-of-year climatology),
the **lagged-rain ridge** (`N_LAGS` days of rain, set in the next cell), and the **conceptual model** you calibrated. The
next line repeats the ridge under a random split, where its held-out days sit between fitted
days. The last line re-runs the comparison that picked `N_LAGS`: it was chosen after comparing
scores like these on these records, which is itself a small look at the test years.

In [ ]:
N_LAGS = 30
RIDGE_LAM = 1.0


def _ridge_forecast(b: int) -> np.ndarray:
    """The lagged-rain ridge, fitted once per catchment on the chronological training days."""
    key = ("ridge", b)
    if key not in _FITS:
        chrono = _chrono_fit(b)
        lags = lagged_matrix(PRECIP[b], N_LAGS)
        w = ridge_fit(lags[chrono["train_idx"]], FLOW[b][chrono["train_idx"]], RIDGE_LAM)
        _FITS[key] = dict(pred=ridge_predict(lags, w))
    return _FITS[key]["pred"]


def _demo_baseline() -> None:
    for b in range(N_BASINS):
        chrono = _chrono_fit(b)
        tr, te = chrono["train_idx"], chrono["test_idx"]
        obs = FLOW[b]
        lags = lagged_matrix(PRECIP[b], N_LAGS)
        ridge = _ridge_forecast(b)
        calendar = doy_climatology(DOY[tr], obs[tr])[DOY[te] - 1]
        rows = (("calendar (fitted years)", calendar), ("lagged-rain ridge", ridge[te]),
                ("conceptual model", chrono["sim"][te]))
        print(f"{STATION_ID[b]}, unseen years:")
        for name, sim in rows:
            print(f"   {name:<24} NSE {nse(obs[te], sim):+.3f} · KGE {kge(obs[te], sim)[0]:+.3f} · "
                  f"skill vs calendar {benchmark_skill(obs[te], sim, calendar):+.3f}")
        train_r, test_r = random_split(N_DAYS, TEST_FRAC, start=WARMUP_DAYS)
        w_r = ridge_fit(lags[train_r], obs[train_r], RIDGE_LAM)
        ridge_random = nse(obs[test_r], ridge_predict(lags, w_r)[test_r])
        print(f"   lagged-rain ridge under a random split: NSE {ridge_random:+.3f} "
              f"(chronological: {nse(obs[te], ridge[te]):+.3f})")
        windows = []
        for n_lags in (7, 30, 90):
            lags_n = lagged_matrix(PRECIP[b], n_lags)
            w_n = ridge_fit(lags_n[tr], obs[tr], RIDGE_LAM)
            windows.append(f"{n_lags} days {nse(obs[te], ridge_predict(lags_n, w_n)[te]):+.3f}")
        print("   the ridge's unseen-year NSE with other rain windows: " + ", ".join(windows))
        RESULTS[b].update(ridge_test_nse=nse(obs[te], ridge[te]), ridge_random_nse=ridge_random)


_try("baseline demo", _demo_baseline, needs=tuple(f"exercise {i}" for i in range(1, 9)))

## 11. Exercise 9 — where in the hydrograph is the skill?

One number averages over every day, and squared errors let the biggest days shout. Two
measures separate the parts. `log_nse` is the NSE of `log(flow + eps)`, which weighs a miss
on a trickle as heavily as a miss on a flood (`eps` is one hundredth of the mean observed
flow, so a zero does not blow up the logarithm). `high_flow_bias` is the relative volume
error over the wettest days: over the days on which the OBSERVED flow is at or above its own
98th percentile, `(sum of sim - sum of obs) / sum of obs`. `flow_band_scores()` returns both.

<details><summary>💡 Hint 1 — what to think about</summary>

Both measures are about the observations' own distribution: the wettest days are chosen by
looking at `obs`, never at `sim`. For the log score you already have a function that scores
two series; what must you feed it? The band edge is "at or above", so ties with the
percentile count.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`eps` is a fixed fraction of the observed mean. Add it to both series, take logs, and score
the pair with the NSE. For the bias: find the 98th percentile of `obs` with the default
linear-interpolation percentile, keep the days at or above it, sum simulated and observed
flow over those days, and return `(sim_sum - obs_sum) / obs_sum`. Return a dict with the two
keys named in the docstring.
</details>

In [ ]:
def flow_band_scores(obs: np.ndarray, sim: np.ndarray) -> dict[str, float]:
    """Two scores that look at different parts of the flow record.

    `"log_nse"`: NSE of `log(obs + eps)` against `log(sim + eps)` with `eps = 0.01 * mean(obs)`.
    `"high_flow_bias"`: over the days where `obs >= np.percentile(obs, 98)`, the value
    `(sum(sim) - sum(obs)) / sum(obs)` (negative means the model under-predicts the biggest days).

    Example:
        >>> obs = np.arange(1.0, 102.0)          # 1 .. 101; the 98th percentile is 99.0
        >>> out = flow_band_scores(obs, obs * 0.5)
        >>> round(out["high_flow_bias"], 3)      # days 99, 100, 101 at half the flow
        -0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_flow_bands() -> None:
    obs = np.arange(1.0, 102.0)
    out = flow_band_scores(obs, obs)
    assert isinstance(out, dict) and set(out) == {"log_nse", "high_flow_bias"}, (
        "return a dict with exactly the keys 'log_nse' and 'high_flow_bias'"
    )
    assert abs(out["log_nse"] - 1.0) < 1e-12 and abs(out["high_flow_bias"]) < 1e-12, "a perfect simulation scores 1 and has no bias"
    out = flow_band_scores(obs, 0.5 * obs)
    assert abs(out["high_flow_bias"] - (-0.5)) < 1e-9, (
        f"halving every flow must give a high-flow bias of -0.5, got {out['high_flow_bias']:.3f}"
    )
    sim = obs.copy()
    sim[:90] = 1.0
    assert abs(flow_band_scores(obs, sim)["high_flow_bias"]) < 1e-12, (
        "the wettest days are chosen from the OBSERVED flow: errors on the other days must not enter"
    )
    out = flow_band_scores(np.array([0.0, 0.0, 0.0, 10.0]), np.array([0.0, 0.0, 0.0, 10.0]))
    assert np.isfinite(out["log_nse"]), "a zero flow must not blow up the log score: add eps first"


_try("exercise 9", _check_flow_bands)

The ledger, on the unseen years. For each catchment: how each forecaster scores on the whole
record (NSE), on the low end (log NSE) and on the biggest days (high-flow bias).

In [ ]:
def _demo_bands() -> None:
    for b in range(N_BASINS):
        chrono = _chrono_fit(b)
        te, obs = chrono["test_idx"], FLOW[b]
        calendar = doy_climatology(DOY[chrono["train_idx"]], obs[chrono["train_idx"]])[DOY[te] - 1]
        print(f"{STATION_ID[b]}, unseen years:")
        for name, sim in (("calendar", calendar),
                          ("lagged-rain ridge", _ridge_forecast(b)[te]),
                          ("conceptual model", chrono["sim"][te])):
            bands = flow_band_scores(obs[te], sim)
            print(f"   {name:<20} NSE {nse(obs[te], sim):+.3f} · log NSE {bands['log_nse']:+.3f} · "
                  f"high-flow bias {bands['high_flow_bias']:+.0%}")
            if name == "conceptual model":
                RESULTS[b].update(model_log_nse=bands["log_nse"], model_high_flow_bias=bands["high_flow_bias"])
        peak_share = obs[te][obs[te] >= np.percentile(obs[te], 98.0)].sum() / obs[te].sum()
        print(f"   the wettest 2% of days carry {peak_share:.0%} of all the flow in these years")


_try("bands demo", _demo_bands, needs=tuple(f"exercise {i}" for i in range(1, 10)))

## 12. Self-check

1. A colleague reports "NSE 0.6, so the model is good" for a river whose flow is strongly
   seasonal. What is the most useful thing to ask for next?
   - (a) The same score computed on the fitted years, which is always more reliable
   - (b) The score of a day-of-year climatology on the same days, so the 0.6 can be read
     against a benchmark that already knows the seasons
   - (c) The number of parameters, because more parameters mean a better score
   - (d) A Kling-Gupta score, which is always higher than NSE

2. You fit a model on a random half of the days and score it on the other half. Why can that
   score not stand in for the skill of a forecast of years to come?
   - (a) Random days are always harder to predict than consecutive ones
   - (b) Held-out days are not the model's fault, so the score is exact
   - (c) Random selection removes the seasonal cycle from the data
   - (d) The fit has already seen days from every year it is scored on, and their neighbours:
     whatever a year shares across its days leaks into the score, and for anything that uses
     nearby flows, the flow itself

3. The routing store in the model is best described as:
   - (a) A slow store that spreads a storm's water over the days and weeks after the rain stopped
   - (b) A correction that is only needed when PET is large
   - (c) A place where the model stores its parameters between calibrations
   - (d) A device that guarantees the simulation is never negative

4. A model scores well on total flow but its high-flow bias is strongly negative. What does
   that tell you?
   - (a) The model is unbiased overall, so it is safe for flood forecasting
   - (b) The model over-predicts low flows
   - (c) The model under-predicts the largest days, which is exactly what a flood warning needs;
     one summary score hid it
   - (d) The observations are wrong on the wettest days

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_nse),
                              ("exercise 2", _check_kge),
                              ("exercise 3", _check_benchmarks),
                              ("exercise 4", _check_unit_hydrographs),
                              ("exercise 5", _check_gr4j_simulate),
                              ("exercise 6", _check_pattern_search),
                              ("exercise 7", _check_holdout),
                              ("exercise 8", _check_baseline),
                              ("exercise 9", _check_flow_bands)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.